In [1]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr
# import gemini
#import google.generativeai
from google import genai
from google.genai import types

In [2]:
# imports for langchain

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [3]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gemini-2.5-flash"
db_name = "vector_db"

In [4]:
# Load environment variables in a file called .env

load_dotenv()
api_key = os.getenv('GEMINI_API_KEY', 'your-key-if-not-using-env')
os.environ["GOOGLE_API_KEY"] = api_key

In [5]:
#google.generativeai.configure()
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(api_version='v1alpha')
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [6]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase

folders = glob.glob("knowledge-base/*")

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [ ]:
# Optional encoding fix for some environments
text_loader_kwargs = {'encoding': 'utf-8'}
# For Windows fallback:
# text_loader_kwargs = {'autodetect_encoding': True}

documents = []

# Recursively find all markdown files under eigen-global
all_md_files = glob.glob("goldisands/**/*.md", recursive=True)

for file_path in all_md_files:
    loader = TextLoader(file_path, **text_loader_kwargs)
    loaded_docs = loader.load()
    for doc in loaded_docs:
        # Add custom metadata based on parent directory name
        doc.metadata["doc_type"] = os.path.basename(os.path.dirname(file_path))
        documents.append(doc)

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [9]:
len(chunks)

28

In [10]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: eigen-global


In [11]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Check if a Chroma Datastore already exists - if so, delete the collection to start from scratch

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Create our Chroma vectorstore!

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 28 documents


In [12]:
# Get one vector and find how many dimensions it has

collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

The vectors have 768 dimensions


RAG pipeline using langchain

In [13]:
# create a new Chat with ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

C:\Users\oshad\AppData\Local\Temp\ipykernel_37208\4130109764.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)


In [14]:
query = "Can you describe Eigen global in a few sentences"
result = conversation_chain.invoke({"question":query})
print(result["answer"])

EigenGlobal is a forward-thinking company dedicated to transforming businesses through innovative digital solutions. They specialize in immersive technologies, aiming to empower organizations by enabling informed decision-making and driving operational excellence. Their focus is on enhancing efficiency and fostering innovation to help businesses navigate and thrive in the digital landscape.


In [15]:
# set up a new conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

Gradio User Interface

In [16]:
def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]

In [17]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
